#  Franchise Supply Chain & Fulfillment Exploration
### Exploratory Data Analysis & Quality Validation Pipeline

This notebook serves as the initial validation layer for the enterprise retail dataset. Before compiling production-grade SQL views or designing executive BI interfaces, we leverage Python to ingest raw transactional files, enforce data types, calculate operational KPIs, and flag anomalous data entries.

## Step 1: Ingestion, Relational Merging, & Datetime Engineering
We ingest the distinct orders and items transactional data streams, execute a relational inner join, and parse raw string timestamps into explicit datetime objects to calculate true **Delivery Lead Time** metrics.

In [ ]:
import pandas as pd

# Load transactional logs
orders = pd.read_csv('../data/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/olist_order_items_dataset.csv')

# Merge datasets to create a core transactional dataframe
df = pd.merge(order_items, orders, on='order_id', how='inner')

# Strict Datetime Parsing
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])

# Drop rows where delivery is incomplete to prevent statistical skew
df = df.dropna(subset=['order_delivered_customer_date'])

# Programmatic Lead Time Calculation
df['Delivery_Lead_Time_Days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

# Sanity Check Audit: Look at top operational latencies
print(df[['order_id', 'Delivery_Lead_Time_Days']].head())

## Step 2: Operational Outlier Auditing & Data Cleansing
Logistical databases frequently contain systemic system clock anomalies (e.g., negative transit days caused by misaligned scans). This step programmatically isolates these exceptions, audits the skew risk, and outputs a refined dataset for downstream BI architecture.

In [ ]:
# Outlier Detection: Identify impossible negative latencies
data_anomalies = df[df['Delivery_Lead_Time_Days'] < 0]
print(f"Alert: Detected {len(data_anomalies)} rows with negative delivery timelines.")

# Cleanse dataset to ensure downstream reporting layer integrity
df_clean = df[df['Delivery_Lead_Time_Days'] >= 0]

# Generate high-level descriptive statistics for regional managers
logistics_summary = df_clean.groupby('order_id')['Delivery_Lead_Time_Days'].mean()
print(logistics_summary.describe())